# 04 — Protótipo do Dashboard

Teste dos gráficos e layout antes de implementar no Streamlit.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import plotly.io as pio

from src.viz import (
    serie_temporal_plotly, barra_cobertura_uf, scatter_cobertura_desfecho,
    barra_faixa_etaria, pizza_sexo, barra_raca, heatmap_regiao_mes,
    treemap_gaps,
)
from src.analysis import cobertura_por_uf, cobertura_por_faixa_etaria, cobertura_por_sexo, cobertura_por_raca

pio.renderers.default = "notebook"

In [ ]:
# Carregar dados
try:
    df = pd.read_parquet("../data/processed/pni_influenza.parquet")
    doses_agg = pd.read_csv("../data/processed/doses_agg_uf_mes.csv")
    print("Dados carregados do cache")
except FileNotFoundError:
    print("Execute o notebook 02_transform primeiro")
    raise

In [ ]:
# Série temporal
serie = df.groupby(["ano", "mes"]).size().reset_index(name="total_doses")
serie["mes_ano"] = serie["ano"].astype(str) + "-" + serie["mes"].astype(str).str.zfill(2)
fig = serie_temporal_plotly(serie)
fig.show()

In [ ]:
# Mapa de cobertura por UF
cov_uf = cobertura_por_uf(doses_agg)
fig = barra_cobertura_uf(cov_uf)
fig.show()

In [ ]:
# Faixa etária
cov_fe = cobertura_por_faixa_etaria(df)
fig = barra_faixa_etaria(cov_fe)
fig.show()

In [ ]:
# Sexo
cov_sexo = cobertura_por_sexo(df)
fig = pizza_sexo(cov_sexo)
fig.show()

In [ ]:
# Raça
cov_raca = cobertura_por_raca(df)
fig = barra_raca(cov_raca)
fig.show()

In [ ]:
# Heatmap região × mês
from src.transform import adiciona_regiao
df_com_regiao = adiciona_regiao(df)
heatmap_data = df_com_regiao.groupby(["regiao", "mes"]).size().reset_index(name="total_doses")
fig = heatmap_regiao_mes(heatmap_data)
fig.show()

In [ ]:
# Exportar gráficos para o relatório
import os
os.makedirs("../relatorios/graficos", exist_ok=True)

fig.write_image("../relatorios/graficos/serie_temporal.png", width=1200, height=600)
print("Gráfico exportado para relatorios/graficos/")

---
**Dashboard completo**: `streamlit run src/dashboard.py`